# 📊 AI-Assisted Trading Risk Manager
**Sentiment → Risk Models → Portfolio Optimisation → Trailing Stops → Monte Carlo**

**Install dependencies:**
```bash
pip install -e ".[market,notebook]"   # yfinance + curl_cffi SSL fallbacks (see src/market/adapters/yfinance.py)
# or: pip install yfinance curl_cffi transformers torch numpy scipy plotly arch hmmlearn
```

> **All tunable parameters live in the two config cells below.**  
> You should rarely need to touch anything else.


## ⚙️ Master Configuration
*Edit anything here — every downstream cell reads from these variables.*

In [1]:
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Repo root on path (run notebook from repo root or notebooks/)
_nb_root = Path.cwd()
if not (_nb_root / 'src').is_dir():
    _nb_root = _nb_root.parent
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from src.market.adapters.yfinance import (
    _make_curl_session,
    _yfinance_session,
    clamp_download_dates,
    load_yfinance_market,
)
from src.utils.config import MarketConfig

from notebooks.trading_notebook_utils import (
    aggregate_portfolio_paths,
    exposure_summary,
    gbm_paths,
    optimize_max_sharpe_gross,
    optimize_partial_weights,
    resolve_weights,
    side_for_ticker,
    simulate_trailing_stops,
)


def yfinance_sessions():
    """Session strategies matching src/market/adapters/yfinance.py."""
    import os

    strategies = [
        ('curl_cffi', _yfinance_session()),
        ('yfinance_default', None),
        ('curl_cffi_no_verify', _make_curl_session(verify=False)),
    ]
    if os.getenv('YFINANCE_SSL_VERIFY', '1').lower() in ('0', 'false', 'no'):
        no_verify = _make_curl_session(verify=False)
        strategies = [('curl_cffi_no_verify', no_verify)] + [
            s for s in strategies if s[0] != 'curl_cffi_no_verify'
        ]
    return strategies


def download_close_prices(tickers, start, end):
    """Download adjusted close prices (wide DataFrame) with SSL fallbacks."""
    start_c, end_excl = clamp_download_dates(start, end)
    rolling = max((date.fromisoformat(end_excl) - date.fromisoformat(start_c)).days, 365)
    cfg = MarketConfig(
        tickers=list(tickers),
        start_date=start,
        end_date=end,
        rolling_days=rolling,
    )
    df = load_yfinance_market(cfg, prefer_period=False).collect().to_pandas()
    if df.empty:
        return pd.DataFrame(columns=list(tickers))
    wide = df.pivot(index='timestamp', columns='ticker', values='close')
    wide.index = pd.to_datetime(wide.index)
    if getattr(wide.index, 'tz', None) is not None:
        wide.index = wide.index.tz_localize(None)
    return wide.sort_index()

# ─────────────────────────────────────────────────────────────────────────────
# 1. PORTFOLIO
# ─────────────────────────────────────────────────────────────────────────────
TICKERS   = ["SMH", "MSFT", "NVDA", "USO", "IWM", "BABA", "PDD"]
START     = '2020-01-01'
END       = '2026-05-22'
RISK_FREE = 0.05     # annual risk-free rate (decimal)

# Gross budget: sum(|weight|) = 1. Negative weight = short (requires ALLOW_SHORTS).
ALLOW_SHORTS = True
MAX_GROSS_PER_TICKER = 0.50
POSITION_SIDES = {'SMH': 'long', 'MSFT': 'long', 'NVDA': 'long', 'USO': 'short', 'IWM': 'short', 'BABA': 'short', 'PDD': 'long'}  # e.g. {'QQQ': 'short'} — force sign after optimization

# Partial weights: fix some tickers, optimize the rest (mode='partial')
# sum(|ANCHOR_WEIGHTS|) must be < 1 so free tickers have budget left.
ANCHOR_WEIGHTS = {
    'CVX': 0.35,
    'GLD': 0.20,
}

SHOW_PER_TICKER_MC = False

# ── Portfolio weighting ───────────────────────────────────────────────────
# 'equal'     → 1/N gross weight on every ticker
# 'manual'    → use MANUAL_WEIGHTS below (signed ok if ALLOW_SHORTS)
# 'partial'   → fix ANCHOR_WEIGHTS, optimize the rest (run Phase 3 first)
# 'optimised' → use max-Sharpe weights from Phase 3
#               (run Phase 3 first, then re-run Phase 2b onward)
PORTFOLIO_WEIGHTING = 'optimised'

# Used only when PORTFOLIO_WEIGHTING = 'manual'. sum(|w|) should be 1 (auto-normalized if not).
MANUAL_WEIGHTS = {
    'CVX': 0.35,
    'XLE': 0.25,
    'QQQ': 0.05,
    'GLD': 0.20,
    'GS':  0.15,
}

# ─────────────────────────────────────────────────────────────────────────────
# 2. SENTIMENT
# ─────────────────────────────────────────────────────────────────────────────
MAX_HEADLINES_PER_TICKER = 10

# ─────────────────────────────────────────────────────────────────────────────
# 3. RISK MODELS  (all run on the weighted portfolio return series)
# ─────────────────────────────────────────────────────────────────────────────
ROLLING_WINDOW        = 21             # days for rolling vol window
EWMA_SPAN             = 21             # EWMA span (λ ≈ 1 - 2/(span+1))
GARCH_P               = 1
GARCH_Q               = 1
VAR_CONFIDENCE_LEVELS = (0.95, 0.99)
VAR_SIM_SIZE          = 100_000

# ─────────────────────────────────────────────────────────────────────────────
# 4. REGIME DETECTION (HMM) — runs on portfolio returns
# ─────────────────────────────────────────────────────────────────────────────
HMM_N_REGIMES = 3
HMM_N_ITER    = 200

# ─────────────────────────────────────────────────────────────────────────────
# 5. PORTFOLIO OPTIMISATION
# ─────────────────────────────────────────────────────────────────────────────
N_SIM_EF               = 3_000
SENTIMENT_RETURN_BOOST = 0.01   # expected-return nudge per unit of sentiment

# ─────────────────────────────────────────────────────────────────────────────
# 6. MONTE CARLO
# ─────────────────────────────────────────────────────────────────────────────
HORIZON               = 252
N_PATHS               = 10_000
PORT_VAL              = 1_000_000   # starting portfolio value ($)
RANDOM_SEED           = 42
PLOT_SAMPLE_PATHS     = 200
SENTIMENT_DRIFT_NUDGE = 0.0002      # daily drift nudge per unit of sentiment

# Regime GBM parameters — ANNUAL drift & vol (converted to daily inside MC)
REGIME_PARAMS = {
    'Bull 🟢':    {'drift':  0.12, 'vol': 0.12},
    'Neutral ⚪': {'drift':  0.04, 'vol': 0.18},
    'Bear 🔴':    {'drift': -0.08, 'vol': 0.28},
}

# ─────────────────────────────────────────────────────────────────────────────
# 7. STRESS TEST
# ─────────────────────────────────────────────────────────────────────────────
N_PATHS_STRESS = 5_000

# Each scenario adjusts two GBM parameters relative to the base regime:
#   drift_shock  — annual return change (e.g. -0.10 = loses 10% annual return)
#   vol_mult     — multiplier on base vol  (e.g.  2.0 = twice as volatile)
# See the "How to set scenarios" markdown cell below for guidance.
SCENARIOS = {
    'Base case':         {'drift_shock':  0.00, 'vol_mult': 1.0},
    'Oil −20%':          {'drift_shock': -0.06, 'vol_mult': 1.4},
    'Rates +100 bps':    {'drift_shock': -0.03, 'vol_mult': 1.2},
    'Market crash −30%': {'drift_shock': -0.25, 'vol_mult': 2.5},
}

print('✅ Master config loaded (yfinance via src/market/adapters/yfinance.py).')


✅ Master config loaded (yfinance via src/market/adapters/yfinance.py).


## ⚙️ Trailing Stop Configuration
Each **ticker** has 3 tranches (⅓ each), with levels from **that ticker's** sentiment.
A stop watches drawdown from the running peak (position MTM); shorts use inverted GBM paths.

| Mode | How to activate |
|---|---|
| **Sentiment-derived** *(default)* | `USE_MANUAL = False` — per-ticker after Phase 1 |
| **Manual override** | `USE_MANUAL = True` and fill `MANUAL_STOP_LEVELS` / `_FRACTIONS` |


In [2]:
USE_MANUAL = False

# ── Manual override ───────────────────────────────────────────────────────
MANUAL_STOP_LEVELS    = [-0.05, -0.10, -0.15]  # drawdown from peak (negative %)
MANUAL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]  # fraction of remaining position to exit

# ── Sentiment thresholds ──────────────────────────────────────────────────
SENTIMENT_BULL_THRESHOLD =  0.3
SENTIMENT_BEAR_THRESHOLD = -0.3

# ── Auto-derived levels per sentiment regime ──────────────────────────────
BULL_STOP_LEVELS    = [-0.08, -0.14, -0.20]   # wide — let winners run
BULL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

NEUTRAL_STOP_LEVELS    = [-0.05, -0.10, -0.15]
NEUTRAL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

BEAR_STOP_LEVELS    = [-0.03, -0.06, -0.10]   # tight — protect capital
BEAR_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

def build_stops(sentiment_score):
    if USE_MANUAL:
        levels, fracs, tag = MANUAL_STOP_LEVELS, MANUAL_STOP_FRACTIONS, '⚙️  Manual'
    elif sentiment_score >= SENTIMENT_BULL_THRESHOLD:
        levels, fracs, tag = BULL_STOP_LEVELS, BULL_STOP_FRACTIONS, '🟢 Bullish-derived'
    elif sentiment_score <= SENTIMENT_BEAR_THRESHOLD:
        levels, fracs, tag = BEAR_STOP_LEVELS, BEAR_STOP_FRACTIONS, '🔴 Bearish-derived'
    else:
        levels, fracs, tag = NEUTRAL_STOP_LEVELS, NEUTRAL_STOP_FRACTIONS, '⚪ Neutral-derived'
    return ([{'level': l, 'exit_fraction': f, 'label': f'Stop {i+1} ({l:.0%})'}
             for i, (l, f) in enumerate(zip(levels, fracs))], tag)

print('✅ Trailing stop config loaded. Run Phase 1 to finalise levels.')


✅ Trailing stop config loaded. Run Phase 1 to finalise levels.


---
## Phase 1 — Financial Sentiment Engine (FinBERT + live news)
Fetches real headlines via `yfinance` (same SSL session fallbacks as `src/market/adapters/yfinance.py`). Falls back to offline headlines if unavailable.


In [3]:
import yfinance as yf
from transformers import pipeline

def get_ticker_headlines(tickers, max_per=MAX_HEADLINES_PER_TICKER):
    """Fetch headlines with the same SSL session fallbacks as market downloads."""
    results = []
    for t in tickers:
        last_err = None
        got = False
        for _label, session in yfinance_sessions():
            try:
                ticker = yf.Ticker(t, session=session) if session else yf.Ticker(t)
                news = ticker.news or []
                batch = []
                for item in news[:max_per]:
                    title = (item.get('content', {}).get('title')
                             if isinstance(item.get('content'), dict)
                             else item.get('title', ''))
                    if title:
                        batch.append({'ticker': t, 'headline': title})
                if batch:
                    results.extend(batch)
                    got = True
                    break
            except Exception as exc:
                last_err = exc
        if not got and last_err is not None:
            print(f'  ⚠️  {t}: {last_err}')
    return results

print('Fetching live headlines...')
live_items = get_ticker_headlines(TICKERS)

# Offline headlines aligned with TICKERS (used when Yahoo news is unreachable)
FALLBACK = [
    {'ticker': t, 'headline': f'{t} beats earnings expectations, raises forward guidance'}
    for t in TICKERS
] + [
    {'ticker': t, 'headline': f'{t} faces sector headwinds amid macro uncertainty'}
    for t in TICKERS
]

headline_items = live_items if live_items else FALLBACK
if not live_items:
    print('⚠️  No live data — using fallback headlines.')
else:
    print(f'✅ {len(live_items)} live headlines fetched.')

headlines = [h['headline'] for h in headline_items]

print('\nLoading FinBERT (~420 MB on first run)...')
sentiment_pipe = pipeline('text-classification', model='ProsusAI/finbert', top_k=None)
results = sentiment_pipe(headlines)

rows = []
for item, scores in zip(headline_items, results):
    if isinstance(scores, dict): scores = [scores]
    sd = {s['label']: s['score'] for s in scores}
    rows.append({'ticker': item['ticker'], 'headline': item['headline'],
                 'positive': round(sd.get('positive',0),3),
                 'negative': round(sd.get('negative',0),3),
                 'neutral':  round(sd.get('neutral', 0),3),
                 'sentiment': max(sd, key=sd.get)})

sentiment_df = pd.DataFrame(rows)
display(sentiment_df)

ticker_sentiment = (sentiment_df.groupby('ticker')
                    .apply(lambda g: g['positive'].mean() - g['negative'].mean())
                    .rename('score').round(3))
portfolio_sentiment = ticker_sentiment.mean()

print('\n📊 Per-ticker sentiment:')
print(ticker_sentiment.to_string())
print(f'\n📰 Portfolio sentiment score: {portfolio_sentiment:+.3f}  (-1 bearish → +1 bullish)')

stops_by_ticker = {}
tags_by_ticker = {}
for t in TICKERS:
    score = float(ticker_sentiment.get(t, 0.0))
    stops_by_ticker[t], tags_by_ticker[t] = build_stops(score)

STOPS, stop_tag = build_stops(portfolio_sentiment)

print('\n🎯 Per-ticker trailing stops (sentiment-derived):')
print(f'  {"Ticker":<6} {"Sent":>7}  {"Regime":<18}  Stop levels')
print('  ' + '─' * 55)
for t in TICKERS:
    lvls = ', '.join(f'{s["level"]:+.0%}' for s in stops_by_ticker[t])
    print(f'  {t:<6} {ticker_sentiment.get(t, 0):>+7.3f}  {tags_by_ticker[t]:<18}  {lvls}')


Fetching live headlines...
✅ 70 live headlines fetched.

Loading FinBERT (~420 MB on first run)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24514.54it/s]


,ticker,headline,positive,negative,neutral,sentiment
0,SMH,VanEck Semiconductor ETF (SMH): Leopold Aschen...,0.011,0.918,0.071,negative
1,SMH,After Three Years of Tracking the AI Capex Cyc...,0.323,0.014,0.664,neutral
2,SMH,The Most-Compared ETFs Right Now — And What Th...,0.045,0.049,0.906,neutral
3,SMH,The AI Memory Shortage Is Just Getting Started...,0.051,0.039,0.911,neutral
4,SMH,Stock Market Today: Nasdaq Leads Rise To Recor...,0.920,0.039,0.041,positive
...,...,...,...,...,...,...
65,PDD,PDD Holdings Inc (PDD) Q1 2026 Earnings Call H...,0.869,0.071,0.060,positive
66,PDD,Equities Mixed Intraday as Markets Track US-Ir...,0.169,0.760,0.072,negative
67,PDD,PDD Holdings shares slide after revenue and pr...,0.009,0.966,0.026,negative
68,PDD,Top Midday Stories: White House Refutes Iran R...,0.011,0.968,0.021,negative



📊 Per-ticker sentiment:
ticker
BABA    0.268
IWM     0.344
MSFT    0.285
NVDA    0.190
PDD    -0.528
SMH     0.008
USO    -0.449

📰 Portfolio sentiment score: +0.017  (-1 bearish → +1 bullish)

🎯 Per-ticker trailing stops (sentiment-derived):
  Ticker    Sent  Regime              Stop levels
  ───────────────────────────────────────────────────────
  SMH     +0.008  ⚪ Neutral-derived   -5%, -10%, -15%
  MSFT    +0.285  ⚪ Neutral-derived   -5%, -10%, -15%
  NVDA    +0.190  ⚪ Neutral-derived   -5%, -10%, -15%
  USO     -0.449  🔴 Bearish-derived   -3%, -6%, -10%
  IWM     +0.344  🟢 Bullish-derived   -8%, -14%, -20%
  BABA    +0.268  ⚪ Neutral-derived   -5%, -10%, -15%
  PDD     -0.528  🔴 Bearish-derived   -3%, -6%, -10%


---
## Phase 2 — Risk Models
### 2a · Download prices & build the portfolio return series

Gross budget: `sum(abs(weight)) = 1`. Negative weights = short.
Modes: `'equal'`, `'manual'`, `'partial'` (anchor + optimize rest), `'optimised'`.

Portfolio charts (vol, VaR, HMM, EF) use **combined** `portfolio_rets`.
Per-ticker tables use individual return series.


In [4]:
prices = download_close_prices(TICKERS, START, END).dropna(how='all')
missing = [t for t in TICKERS if t not in prices.columns or prices[t].dropna().empty]
if missing:
    print(f'⚠️  No price data for: {missing}')
prices = prices[[t for t in TICKERS if t in prices.columns]].dropna()
rets   = prices.pct_change().dropna()
print(f'Downloaded {len(prices)} days for {prices.shape[1]}/{len(TICKERS)} tickers.')
if prices.empty:
    raise ValueError(
        'No market data downloaded. Install: pip install -e ".[market]" '
        'or set YFINANCE_SSL_VERIFY=0 for dev (see src/market/adapters/yfinance.py).'
    )

mu_daily = rets.mean()
cov_daily = rets.cov()

_kw = dict(
    allow_shorts=ALLOW_SHORTS,
    max_gross_per_ticker=MAX_GROSS_PER_TICKER,
    position_sides=POSITION_SIDES,
    manual_weights=MANUAL_WEIGHTS,
    anchor_weights=ANCHOR_WEIGHTS,
    risk_free=RISK_FREE,
)

try:
    if PORTFOLIO_WEIGHTING == 'optimised':
        weights = resolve_weights('optimised', TICKERS, w_optimised=w_sharpe, **_kw)
    elif PORTFOLIO_WEIGHTING == 'partial':
        try:
            weights = resolve_weights('partial', TICKERS, w_optimised=w_sharpe, **_kw)
        except NameError:
            print('⚠️  w_sharpe not yet computed — optimizing partial weights inline.')
            weights = resolve_weights(
                'partial',
                TICKERS,
                mean_returns=mu_daily.values * 252,
                cov=cov_daily.values * 252,
                **_kw,
            )
    else:
        weights = resolve_weights(PORTFOLIO_WEIGHTING, TICKERS, **_kw)
except NameError:
    print('⚠️  w_sharpe not yet computed — falling back to equal gross weights.')
    print('   Run Phase 3 first for optimised/partial, then re-run this cell.')
    weights = resolve_weights('equal', TICKERS, **_kw)

weights = np.asarray(weights, dtype=float)
exp = exposure_summary(weights)
portfolio_rets = (rets * weights).sum(axis=1)

print(f'\n📐 Portfolio weights ({PORTFOLIO_WEIGHTING}):')
for t, w in zip(TICKERS, weights):
    side = 'short' if w < 0 else 'long'
    print(f'  {t}: {w:+.1%}  ({side})')
print(f'  Gross: {exp["gross"]:.2%}  |  Net: {exp["net"]:+.2%}  |  Long: {exp["long"]:+.2%}  |  Short: {exp["short"]:+.2%}')
print(f'\nPortfolio return series: {len(portfolio_rets)} days')
print(f'  Annual mean : {portfolio_rets.mean()*252:.2%}')
print(f'  Annual vol  : {portfolio_rets.std()*np.sqrt(252):.2%}')


[INFO] Downloading market data via yfinance
[INFO] yfinance strategy=yfinance_default tickers=7/7
Downloaded 1256 days for 7/7 tickers.
⚠️  w_sharpe not yet computed — falling back to equal gross weights.
   Run Phase 3 first for optimised/partial, then re-run this cell.

📐 Portfolio weights (optimised):
  SMH: +14.3%  (long)
  MSFT: +14.3%  (long)
  NVDA: +14.3%  (long)
  USO: -14.3%  (short)
  IWM: -14.3%  (short)
  BABA: -14.3%  (short)
  PDD: +14.3%  (long)
  Gross: 100.00%  |  Net: +14.29%  |  Long: +57.14%  |  Short: -42.86%

Portfolio return series: 1255 days
  Annual mean : 13.55%
  Annual vol  : 15.92%


### 2a-iii · Per-ticker risk snapshot
Position-level metrics (each name traded separately). Portfolio charts below still use the combined `portfolio_rets` series.


In [5]:
def _ticker_risk_row(t):
    r = rets[t].dropna()
    if r.empty:
        return None
    ann_mu = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = (ann_mu - RISK_FREE) / ann_vol if ann_vol > 1e-12 else np.nan
    var95 = np.percentile(r, 5)
    cvar95 = r[r <= var95].mean() if (r <= var95).any() else var95
    wi = float(weights[TICKERS.index(t)])
    return {
        'ticker': t,
        'side': 'short' if wi < 0 else 'long',
        'weight': wi,
        'gross': abs(wi),
        'ann_return': ann_mu,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'VaR_95_daily': var95,
        'CVaR_95_daily': cvar95,
    }

rows = [_ticker_risk_row(t) for t in TICKERS]
risk_by_ticker = pd.DataFrame([x for x in rows if x])
display(risk_by_ticker.style.format({
    'weight': '{:+.1%}', 'gross': '{:.1%}',
    'ann_return': '{:.2%}', 'ann_vol': '{:.2%}', 'sharpe': '{:.2f}',
    'VaR_95_daily': '{:.2%}', 'CVaR_95_daily': '{:.2%}',
}))


,ticker,side,weight,gross,ann_return,ann_vol,sharpe,VaR_95_daily,CVaR_95_daily
0,SMH,long,+14.3%,14.3%,38.39%,34.97%,0.95,-3.43%,-4.77%
1,MSFT,long,+14.3%,14.3%,16.98%,26.54%,0.45,-2.66%,-3.75%
2,NVDA,long,+14.3%,14.3%,65.57%,51.63%,1.17,-4.75%,-6.66%
3,USO,short,-14.3%,14.3%,28.06%,36.04%,0.64,-3.49%,-5.17%
4,IWM,short,-14.3%,14.3%,8.46%,22.51%,0.15,-2.19%,-3.00%
5,BABA,short,-14.3%,14.3%,2.44%,51.31%,-0.05,-4.50%,-6.52%
6,PDD,long,+14.3%,14.3%,12.84%,68.13%,0.12,-6.02%,-9.16%


### 2a-ii · Historical Portfolio Performance & Correlations

In [6]:

# ── Cumulative portfolio value (historical backtest) ──────────────────────
cumulative_port = (1 + portfolio_rets).cumprod() * PORT_VAL
cumulative_indiv = (1 + rets).cumprod() * PORT_VAL

fig = go.Figure()
for t in TICKERS:
    fig.add_trace(go.Scatter(x=cumulative_indiv.index, y=cumulative_indiv[t],
                             name=t, line=dict(width=1), opacity=0.6))
fig.add_trace(go.Scatter(x=cumulative_port.index, y=cumulative_port,
                         name=f'Portfolio ({PORTFOLIO_WEIGHTING})',
                         line=dict(width=3, color='white')))
fig.update_layout(title=f'Historical Cumulative Value — ${PORT_VAL:,.0f} starting capital',
                  xaxis_title='Date', yaxis_title='Portfolio Value ($)',
                  template='plotly_dark')
fig.show()

# ── Correlation heatmap ───────────────────────────────────────────────────
corr = rets.corr().round(2)
fig2 = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=corr.values.round(2), texttemplate='%{text}',
    colorbar=dict(title='Correlation')))
fig2.update_layout(title='Return Correlation Matrix', template='plotly_dark')
fig2.show()

# ── Max drawdown per asset & portfolio ───────────────────────────────────
def max_drawdown(s):
    roll_max = s.cummax()
    dd = (s - roll_max) / roll_max
    return dd.min()

print('\n📉 Maximum Historical Drawdown:')
for t in TICKERS:
    cum = (1 + rets[t]).cumprod()
    print(f'  {t:6}  {max_drawdown(cum):.2%}')
print(f'  {"Portfolio":6}  {max_drawdown(cumulative_port):.2%}')



📉 Maximum Historical Drawdown:
  SMH     -45.30%
  MSFT    -37.15%
  NVDA    -66.34%
  USO     -36.23%
  IWM     -31.91%
  BABA    -72.48%
  PDD     -81.75%
  Portfolio  -20.23%


### 2b · Portfolio Volatility Forecasting (Rolling, EWMA, GARCH)

In [7]:
from arch import arch_model

r = portfolio_rets * 100   # scale for GARCH stability

roll_vol = r.rolling(ROLLING_WINDOW).std() * np.sqrt(252) / 100
ewma_vol = r.ewm(span=EWMA_SPAN).std()     * np.sqrt(252) / 100

_min_garch = ROLLING_WINDOW + 30
if len(r.dropna()) < _min_garch:
    raise ValueError(
        f'Need at least {_min_garch} return days for GARCH (have {len(r.dropna())}). '
        'Re-run Phase 2a after market download succeeds.'
    )
garch_fit = arch_model(r, vol='Garch', p=GARCH_P, q=GARCH_Q, rescale=False).fit(disp='off')
garch_vol = garch_fit.conditional_volatility * np.sqrt(252) / 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_vol.index, y=roll_vol,  name=f'Rolling {ROLLING_WINDOW}d'))
fig.add_trace(go.Scatter(x=ewma_vol.index, y=ewma_vol,  name=f'EWMA (span={EWMA_SPAN})'))
fig.add_trace(go.Scatter(x=garch_vol.index, y=garch_vol, name=f'GARCH({GARCH_P},{GARCH_Q})'))
fig.update_layout(title='Portfolio — Annualised Volatility Forecasts',
                  yaxis_tickformat='.0%', template='plotly_dark')
fig.show()
print(f'Latest GARCH portfolio vol: {garch_vol.iloc[-1]:.2%}')


Latest GARCH portfolio vol: 15.27%


### 2b-ii · Rolling Portfolio Metrics (Sharpe, Drawdown)

In [8]:

ROLLING_METRICS_WINDOW = 63   # ~1 quarter; override here if needed

roll_ret  = portfolio_rets.rolling(ROLLING_METRICS_WINDOW).mean() * 252
roll_vol2 = portfolio_rets.rolling(ROLLING_METRICS_WINDOW).std()  * np.sqrt(252)
roll_sharpe = (roll_ret - RISK_FREE) / roll_vol2

# Rolling drawdown on cumulative portfolio
cum_port  = (1 + portfolio_rets).cumprod()
roll_peak = cum_port.cummax()
roll_dd   = (cum_port - roll_peak) / roll_peak

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_sharpe.index, y=roll_sharpe,
                         name=f'Rolling {ROLLING_METRICS_WINDOW}d Sharpe',
                         line=dict(color='gold', width=2)))
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.add_hline(y=1, line_dash='dot',  line_color='lime', annotation_text='Sharpe = 1')
fig.update_layout(title=f'Portfolio Rolling {ROLLING_METRICS_WINDOW}-Day Sharpe Ratio',
                  yaxis_title='Sharpe', template='plotly_dark')
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=roll_dd.index, y=roll_dd,
                          fill='tozeroy', name='Drawdown',
                          line=dict(color='red', width=1)))
fig2.update_layout(title='Portfolio Historical Drawdown',
                   yaxis_title='Drawdown', yaxis_tickformat='.0%',
                   template='plotly_dark')
fig2.show()

print(f'Rolling {ROLLING_METRICS_WINDOW}d Sharpe  — current: {roll_sharpe.iloc[-1]:.2f} '
      f'| max: {roll_sharpe.max():.2f} | min: {roll_sharpe.min():.2f}')
print(f'Max historical drawdown (portfolio): {roll_dd.min():.2%}')
print(f'Current drawdown from peak:          {roll_dd.iloc[-1]:.2%}')


Rolling 63d Sharpe  — current: 0.18 | max: 5.74 | min: -4.31
Max historical drawdown (portfolio): -20.23%
Current drawdown from peak:          -7.46%


### 2c · Portfolio VaR & CVaR

In [9]:
from scipy import stats

def compute_risk(returns, confidence_levels=VAR_CONFIDENCE_LEVELS):
    rows, mu, sigma = [], returns.mean(), returns.std()
    sim = np.random.normal(mu, sigma, VAR_SIM_SIZE)
    for cl in confidence_levels:
        a   = 1 - cl
        hv  = -np.percentile(returns, a*100)
        hcv = -returns[returns <= -hv].mean()
        pv  = -(mu + stats.norm.ppf(a)*sigma)
        pcv = -(mu - sigma*stats.norm.pdf(stats.norm.ppf(a))/a)
        mv  = -np.percentile(sim, a*100)
        mcv = -sim[sim <= -mv].mean()
        rows.append({'Confidence': f'{cl:.0%}',
                     'Hist VaR': f'{hv:.2%}',  'Hist CVaR': f'{hcv:.2%}',
                     'Param VaR': f'{pv:.2%}', 'Param CVaR': f'{pcv:.2%}',
                     'MC VaR': f'{mv:.2%}',    'MC CVaR': f'{mcv:.2%}'})
    return pd.DataFrame(rows)

display(compute_risk(portfolio_rets))


,Confidence,Hist VaR,Hist CVaR,Param VaR,Param CVaR,MC VaR,MC CVaR
0,95%,1.44%,2.13%,1.60%,2.01%,1.59%,2.01%
1,99%,2.46%,2.94%,2.28%,2.62%,2.27%,2.62%


### 2d · Portfolio Regime Detection (HMM)

In [10]:
from hmmlearn.hmm import GaussianHMM

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

r_arr   = portfolio_rets.values.reshape(-1, 1)
hmm     = GaussianHMM(n_components=HMM_N_REGIMES, covariance_type='full',
                      n_iter=HMM_N_ITER, random_state=RANDOM_SEED)
hmm.fit(r_arr)
regimes = hmm.predict(r_arr)
ranking = sorted(range(HMM_N_REGIMES), key=lambda i: hmm.means_[i][0])

regime_name_map = {}
for pos, state in enumerate(ranking):
    if pos == 0:
        regime_name_map[state] = 'Bear 🔴'
    elif pos == len(ranking) - 1:
        regime_name_map[state] = 'Bull 🟢'
    else:
        regime_name_map[state] = 'Neutral ⚪'

regime_labels = pd.Series([regime_name_map[r] for r in regimes], index=portfolio_rets.index)

fig = px.scatter(x=portfolio_rets.index, y=portfolio_rets, color=regime_labels,
                 color_discrete_map={'Bear 🔴':'red','Neutral ⚪':'grey','Bull 🟢':'green'},
                 title=f'Portfolio Daily Returns — HMM Regime Detection ({HMM_N_REGIMES} regimes)',
                 template='plotly_dark')
fig.update_traces(marker_size=3)
fig.show()

current_regime = regime_labels.iloc[-1]
print(f'Current regime: {current_regime}')

print('\n📊 Regime summary:')
for name in ['Bull 🟢', 'Neutral ⚪', 'Bear 🔴']:
    mask = regime_labels == name
    if mask.any():
        r = portfolio_rets[mask]
        print(f'  {name:15}  {mask.mean():.1%} of days  '
              f'| mean ret {r.mean()*252:>+.1%}/yr  '
              f'| vol {r.std()*np.sqrt(252):.1%}/yr')


Model is not converging.  Current: 4025.127102375229 is not greater than 4025.320077804052. Delta is -0.19297542882304697


Current regime: Neutral ⚪

📊 Regime summary:
  Bull 🟢           1.0% of days  | mean ret +196.1%/yr  | vol 56.6%/yr
  Neutral ⚪        49.5% of days  | mean ret +16.9%/yr  | vol 15.2%/yr
  Bear 🔴           49.5% of days  | mean ret +6.4%/yr  | vol 14.7%/yr


---
## Phase 3 — Portfolio Optimisation (Efficient Frontier)

Finds the **max-Sharpe** portfolio using your tickers and sentiment-adjusted expected returns.

> 💡 To use these weights in Phase 2, set `PORTFOLIO_WEIGHTING = 'optimised'` in the config  
> and re-run **Phase 2a onward**.


In [11]:
mu_annual  = rets.mean() * 252
cov_annual = rets.cov()  * 252
n          = len(TICKERS)

for t in TICKERS:
    if t in ticker_sentiment.index:
        mu_annual[t] += ticker_sentiment[t] * SENTIMENT_RETURN_BOOST

def portfolio_stats(w):
    ret    = w @ mu_annual
    vol    = np.sqrt(w @ cov_annual @ w)
    sharpe = (ret - RISK_FREE) / vol if vol > 1e-12 else 0.0
    return ret, vol, sharpe

if PORTFOLIO_WEIGHTING == 'partial':
    w_sharpe, ok = optimize_partial_weights(
        mu_annual.values, cov_annual.values, TICKERS, ANCHOR_WEIGHTS,
        risk_free=RISK_FREE, allow_shorts=ALLOW_SHORTS,
        max_gross_per_ticker=MAX_GROSS_PER_TICKER, position_sides=POSITION_SIDES,
    )
else:
    w_sharpe, ok = optimize_max_sharpe_gross(
        mu_annual.values, cov_annual.values,
        risk_free=RISK_FREE, allow_shorts=ALLOW_SHORTS,
        max_gross_per_ticker=MAX_GROSS_PER_TICKER,
    )
    from notebooks.trading_notebook_utils import apply_position_sides
    w_sharpe = apply_position_sides(w_sharpe, TICKERS, POSITION_SIDES)

# Random portfolios on gross simplex
sim_w = np.random.dirichlet(np.ones(n), N_SIM_EF)
if ALLOW_SHORTS:
  signs = np.array([1.0 if POSITION_SIDES.get(t, 'long') == 'long' else -1.0 for t in TICKERS])
  sim_w = sim_w * signs
sim_stats = np.array([portfolio_stats(w) for w in sim_w])

fig = go.Figure()
fig.add_trace(go.Scatter(x=sim_stats[:,1], y=sim_stats[:,0], mode='markers',
    marker=dict(color=sim_stats[:,2], colorscale='Viridis', size=4,
                showscale=True, colorbar=dict(title='Sharpe')),
    name=f'{N_SIM_EF} random portfolios'))
r_opt, v_opt, s_opt = portfolio_stats(w_sharpe)
fig.add_trace(go.Scatter(x=[v_opt], y=[r_opt], mode='markers+text',
    marker=dict(color='red', size=14, symbol='star'),
    text=['Max Sharpe'], textposition='top right', name='Max Sharpe'))
fig.update_layout(title='Efficient Frontier (gross budget, sentiment-enhanced)',
                  xaxis_title='Volatility', yaxis_title='Return',
                  xaxis_tickformat='.0%', yaxis_tickformat='.0%',
                  template='plotly_dark')
fig.show()

exp = exposure_summary(w_sharpe)
print('\n📌 Optimised weights:')
for t, w in zip(TICKERS, w_sharpe):
    print(f'  {t}: {w:+.1%}')
print(f'  Return: {r_opt:.2%}  |  Vol: {v_opt:.2%}  |  Sharpe: {s_opt:.2f}')
print(f'  Gross: {exp["gross"]:.2%}  |  Net: {exp["net"]:+.2%}')
print('\n💡 Re-run Phase 2a to apply optimised/partial weights.')



📌 Optimised weights:
  SMH: +1.5%
  MSFT: +0.0%
  NVDA: +50.0%
  USO: -35.8%
  IWM: -4.6%
  BABA: -8.1%
  PDD: +0.0%
  Return: 22.97%  |  Vol: 27.63%  |  Sharpe: 0.65
  Gross: 100.00%  |  Net: +3.06%

💡 Re-run Phase 2a to apply optimised/partial weights.


---
## Phase 4 — Regime-Aware Monte Carlo with Trailing Stops

GBM drift & vol come from the **current HMM regime** of the portfolio.  
The 3-tranche trailing stop engine runs day-by-day on each simulated path.


In [12]:
if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

path_by_ticker = {}
raw_by_ticker = {}
for ti, t in enumerate(TICKERS):
    side = 'short' if weights[ti] < 0 else 'long'
    ann_mu = float(rets[t].mean() * 252)
    ann_vol = float(rets[t].std() * np.sqrt(252))
    rp = REGIME_PARAMS.get(current_regime, REGIME_PARAMS.get('Neutral ⚪', list(REGIME_PARAMS.values())[1]))
    drift = rp['drift'] / 252 + float(ticker_sentiment.get(t, 0)) * SENTIMENT_DRIFT_NUDGE
    vol_d = max(ann_vol, 1e-6) / np.sqrt(252)
    alloc = abs(weights[ti]) * PORT_VAL
    raw = gbm_paths(
        horizon=HORIZON, n_paths=N_PATHS, start_value=alloc,
        drift_daily=drift, vol_daily=vol_d,
        seed=None if RANDOM_SEED is None else RANDOM_SEED + ti,
        side=side,
    )
    raw_by_ticker[t] = raw
    stops_t = stops_by_ticker.get(t, STOPS)
    path_by_ticker[t] = simulate_trailing_stops(raw, stops_t, side='long')

paths_raw = aggregate_portfolio_paths(raw_by_ticker, weights, TICKERS)
paths_stopped = aggregate_portfolio_paths(path_by_ticker, weights, TICKERS)

final_raw = paths_raw[-1]
final_stopped = paths_stopped[-1]
var95_raw = np.percentile(final_raw, 5)
var95_stopped = np.percentile(final_stopped, 5)
days_ax = np.arange(HORIZON + 1)
pct = lambda arr, q: np.percentile(arr, q, axis=1)

fig = go.Figure()
idx = np.random.choice(N_PATHS, min(PLOT_SAMPLE_PATHS, N_PATHS), replace=False)
for i in idx:
    fig.add_trace(go.Scatter(x=days_ax, y=paths_stopped[:, i],
        line=dict(width=0.5, color='steelblue'), name='With stops',
        legendgroup='s', showlegend=bool(i == idx[0]), opacity=0.5))

for q, col, dash, lbl in [(95, 'lime', 'dot', '95th (w/ stops)'), (50, 'white', 'solid', 'Median'), (5, 'red', 'dot', '5th (VaR proxy)')]:
    fig.add_trace(go.Scatter(x=days_ax, y=pct(paths_stopped, q), mode='lines',
        line=dict(color=col, width=2, dash=dash), name=lbl))

fig.update_layout(
    title=(f'Monte Carlo — {N_PATHS:,} paths | Per-ticker stops | Regime: {current_regime}'),
    xaxis_title='Trading days', yaxis_title='Portfolio value ($)',
    yaxis_tickformat='$,.0f', template='plotly_dark')
fig.show()

if SHOW_PER_TICKER_MC:
    fig2 = go.Figure()
    for t in TICKERS:
        fig2.add_trace(go.Scatter(x=days_ax, y=path_by_ticker[t][:, 0], name=t, line=dict(width=1)))
    fig2.update_layout(title='Sample per-ticker stopped paths (path 0)', template='plotly_dark')
    fig2.show()

print(f'\n📉 1-Year Risk Summary | Weights: {PORTFOLIO_WEIGHTING} | Regime: {current_regime}')
print(f'  VaR 95% (stopped aggregate): ${var95_stopped:,.0f}')



📉 1-Year Risk Summary | Weights: optimised | Regime: Neutral ⚪
  VaR 95% (stopped aggregate): $132,747


---
## Stress Test — What the scenarios mean & how to set them

Each scenario tweaks two parameters of the GBM that drives the Monte Carlo:

| Parameter | What it represents | Example |
|---|---|---|
| `drift_shock` | **Annual return penalty** added on top of the base regime drift. Negative = bad news. | `-0.10` = "this shock costs the portfolio 10% annual return" |
| `vol_mult` | **Volatility multiplier** relative to base regime vol. >1 = more turbulent. | `2.0` = "twice as volatile as normal" |

### How to calibrate them from historical events

A useful approach: look at what actually happened during a comparable past episode.

**Drift shock** — take the annualised excess return of your portfolio during the event vs a calm baseline period, e.g.:
- 2022 rate-hike cycle hit a balanced equity portfolio by roughly **−15 to −20% annual return** → `drift_shock = -0.15`
- A sector-specific shock (oil names during 2014−16 oil crash) might be **−30 to −40%** → `drift_shock = -0.35`
- A mild macro headwind might only cost **−3 to −5%** → `drift_shock = -0.03`

**Vol multiplier** — compare the average VIX (or realised vol) during the episode to the calm period:
- VIX doubles from 15 → 30: `vol_mult ≈ 2.0`
- VIX goes from 15 → 45 (2020 COVID crash): `vol_mult ≈ 3.0`
- Mild turbulence (VIX 15 → 20): `vol_mult ≈ 1.3`

### Suggested starting points by event type

```python
# Add / edit entries in SCENARIOS in the config cell
SCENARIOS = {
    'Base case':           {'drift_shock':  0.00, 'vol_mult': 1.0},
    'Rate hike +100 bps':  {'drift_shock': -0.03, 'vol_mult': 1.2},
    'Mild recession':      {'drift_shock': -0.10, 'vol_mult': 1.5},
    'Deep recession':      {'drift_shock': -0.20, 'vol_mult': 2.0},
    'Oil shock −20%':      {'drift_shock': -0.06, 'vol_mult': 1.4},
    'Credit crunch':       {'drift_shock': -0.15, 'vol_mult': 2.2},
    'COVID-style crash':   {'drift_shock': -0.30, 'vol_mult': 3.0},
    'Soft landing':        {'drift_shock':  0.03, 'vol_mult': 0.8},
}
```


In [13]:
if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

# Portfolio-level GBM (regime + portfolio sentiment); independent of per-ticker MC cell.
rp = REGIME_PARAMS.get(current_regime, REGIME_PARAMS.get('Neutral ⚪', list(REGIME_PARAMS.values())[1]))
drift = rp['drift'] / 252 + portfolio_sentiment * SENTIMENT_DRIFT_NUDGE
vol = rp['vol'] / np.sqrt(252)

try:
    stops_sorted
except NameError:
    stops_sorted = sorted(STOPS, key=lambda s: s['level'])

rows = []
for name, params in SCENARIOS.items():
    d  = drift + params['drift_shock'] / 252
    v  = vol   * params['vol_mult']
    lr = (d - 0.5*v**2) + v * np.random.normal(0, 1, (HORIZON, N_PATHS_STRESS))
    dr = np.exp(lr)

    im = np.full(N_PATHS_STRESS, float(PORT_VAL))
    cs = np.zeros(N_PATHS_STRESS)
    pk = np.full(N_PATHS_STRESS, float(PORT_VAL))
    trig = [np.zeros(N_PATHS_STRESS, dtype=bool) for _ in stops_sorted]

    for day in range(HORIZON):
        im *= dr[day]; pk = np.maximum(pk, im)
        dd = np.where(pk > 0, (im-pk)/pk, 0.0)
        for i, stop in enumerate(stops_sorted):
            fire = (~trig[i]) & (dd <= stop['level'])
            if fire.any():
                locked = stop['exit_fraction'] * im[fire]
                cs[fire] += locked; im[fire] -= locked; trig[i][fire] = True

    f = im + cs
    row = {'Scenario': name,
           'Drift shock': f"{params['drift_shock']:>+.0%}/yr",
           'Vol mult':    f"{params['vol_mult']:.1f}×",
           'Median P&L':  f'${np.median(f)-PORT_VAL:>+,.0f}',
           '5th pct P&L': f'${np.percentile(f,5)-PORT_VAL:>+,.0f}',
           'Prob loss':   f'{(f<PORT_VAL).mean():.1%}'}
    for i, stop in enumerate(stops_sorted):
        row[stop['label']] = f'{trig[i].mean():.1%}'
    rows.append(row)

display(pd.DataFrame(rows))


,Scenario,Drift shock,Vol mult,Median P&L,5th pct P&L,Prob loss,Stop 3 (-15%),Stop 2 (-10%),Stop 1 (-5%)
0,Base case,+0%/yr,1.0×,$+494,"$-97,646",49.7%,100.0%,100.0%,100.0%
1,Oil −20%,-6%/yr,1.4×,"$-21,934","$-138,026",59.0%,100.0%,100.0%,100.0%
2,Rates +100 bps,-3%/yr,1.2×,"$-9,182","$-121,405",54.6%,100.0%,100.0%,100.0%
3,Market crash −30%,-25%/yr,2.5×,"$-86,240","$-227,729",73.3%,100.0%,100.0%,100.0%


---
## 📋 Final Summary Dashboard
All key outputs in one place.

In [14]:

print('=' * 65)
print('  📊 TRADING RISK MANAGER — FINAL SUMMARY')
print('=' * 65)

print(f'\n🗂  Portfolio  ({PORTFOLIO_WEIGHTING} weights, {START} → {END})')
for t, w in zip(TICKERS, weights):
    sent = ticker_sentiment.get(t, float('nan'))
    side = 'short' if w < 0 else 'long'
    tag = tags_by_ticker.get(t, stop_tag)
    print(f'   {t:6} {w:>+7.1%}  ({side})   sentiment: {sent:>+.3f}   stops: {tag}')
print(f'   Portfolio sentiment: {portfolio_sentiment:+.3f}')

print(f'\n📈 Historical Performance')
ann_ret = portfolio_rets.mean() * 252
ann_vol = portfolio_rets.std()  * np.sqrt(252)
print(f'   Annual return  : {ann_ret:>+.2%}')
print(f'   Annual vol     : {ann_vol:.2%}')
print(f'   Sharpe ratio   : {(ann_ret - RISK_FREE) / ann_vol:.2f}')
print(f'   Max drawdown   : {roll_dd.min():.2%}')

print(f'\n🎯 Risk Metrics (portfolio, GARCH vol = {garch_vol.iloc[-1]:.2%})')
risk_tbl = compute_risk(portfolio_rets)
print(risk_tbl.to_string(index=False))

print(f'\n🔭 Current Regime  : {current_regime}')
print('   Per-ticker trailing stop levels:')
for t in TICKERS:
    lvls = ', '.join(f'{s["level"]:+.0%}' for s in stops_by_ticker.get(t, STOPS))
    print(f'   {t:6}  {tags_by_ticker.get(t, stop_tag):<18}  {lvls}')

print(f'\n💼 Optimised Max-Sharpe Portfolio')
for t, w in zip(TICKERS, w_sharpe):
    print(f'   {t:6} {w:+.1%}')
print(f'   Expected return: {r_opt:.2%}  |  Vol: {v_opt:.2%}  |  Sharpe: {s_opt:.2f}')

print(f'\n🎲 Monte Carlo ({N_PATHS:,} paths, {HORIZON}d, ${PORT_VAL:,.0f} start)')
print(f'   {"":26}  {"No stops":>12}  {"With stops":>12}')
print('   ' + '─' * 54)
for label, q in [('Median final value', 50), ('95th pct', 95), ('5th pct', 5)]:
    print(f'   {label:26}  ${np.percentile(final_raw,q):>10,.0f}  '
          f'${np.percentile(final_stopped,q):>10,.0f}')
print(f'   {"VaR 95%":26}  ${PORT_VAL-var95_raw:>10,.0f}  '
      f'${PORT_VAL-var95_stopped:>10,.0f}')
print('=' * 65)


  📊 TRADING RISK MANAGER — FINAL SUMMARY

🗂  Portfolio  (optimised weights, 2020-01-01 → 2026-05-22)
   SMH     +14.3%  (long)   sentiment: +0.008   stops: ⚪ Neutral-derived
   MSFT    +14.3%  (long)   sentiment: +0.285   stops: ⚪ Neutral-derived
   NVDA    +14.3%  (long)   sentiment: +0.190   stops: ⚪ Neutral-derived
   USO     -14.3%  (short)   sentiment: -0.449   stops: 🔴 Bearish-derived
   IWM     -14.3%  (short)   sentiment: +0.344   stops: 🟢 Bullish-derived
   BABA    -14.3%  (short)   sentiment: +0.268   stops: ⚪ Neutral-derived
   PDD     +14.3%  (long)   sentiment: -0.528   stops: 🔴 Bearish-derived
   Portfolio sentiment: +0.017

📈 Historical Performance
   Annual return  : +13.55%
   Annual vol     : 15.92%
   Sharpe ratio   : 0.54
   Max drawdown   : -20.23%

🎯 Risk Metrics (portfolio, GARCH vol = 15.27%)
Confidence Hist VaR Hist CVaR Param VaR Param CVaR MC VaR MC CVaR
       95%    1.44%     2.13%     1.60%      2.01%  1.60%   2.01%
       99%    2.46%     2.94%     2.28% 